# Run 1c: finding an edit that is strong, specific and not a yes-bias
Run 1b showed that edits *after* the mediator sentence are mostly a "say Yes" push, while edits **at** the mediator sentence (layer 12) were specific but too weak.

This run edits **only the neutral mediator sentence** ("Asha looked at the road."):
* **token**: the last token of that sentence (as before); **span**: every token of that sentence;
* strengths α = 4, 8, 16 (in units of the true − false difference);
* layers 4, 8, 12, 16;
* real direction vs an unrelated scenario's direction (same length); three filler questions for the yes-bias.

**Pass rule, fixed before running.** A setting passes if, averaged over scenarios: the filler-corrected mediator shift is ≥ 1.0; the real direction beats the unrelated one on it by > 0.5 and in ≥ 70% of scenarios; and the filler shift is < 0.5.
Reads run 1's `results/`. T4, about 20 minutes. **Runtime → Run all**.

In [ ]:
# Mount Google Drive and work inside the see-vs-do folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/see-vs-do")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
# ---------- sentence-wide edits: encode the first and last token of the key sentence
import glob
from scipy.stats import wilcoxon

def encode2(stories, qs, spans):
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    first, last = [], []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0
        first.append(hit[0].item()); last.append(hit[-1].item())
    return enc, first, last

def belief2(stories, qs, spans, edit=None, bs=BS):
    res = []
    for i in range(0, len(stories), bs):
        enc, first, last = encode2(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = torch.zeros(len(first), T)
            for r, (f, l) in enumerate(zip(first, last)):
                if edit["mode"] == "token":
                    mask[r, l] = 1
                else:
                    mask[r, f:l + 1] = 1
            mask = mask.to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

sc = pd.read_csv(f"{OUT}/A_reference_scenarios.csv", index_col=0)
VALID = sc.index[sc.valid].tolist()
DIRS = torch.load(f"{OUT}/B_directions.pt")["dirs"]
X_LAYERS, MODES, ALPHAS_X = [4, 8, 12, 16], ["token", "span"], [4.0, 8.0, 16.0]
LI = {L: LAYERS.index(L) for L in X_LAYERS}
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
EV = ev[ev.scenario_id.isin(VALID)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    EV[f"q_f{i}"] = f
ALL_PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}
OTHER = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
UNREL = {s: DIRS[OTHER[s]] / DIRS[OTHER[s]].norm(dim=-1, keepdim=True) * DIRS[s].norm(dim=-1, keepdim=True) for s in VALID}
MAPS = {"real": DIRS, "unrelated": UNREL}
xsc = pd.read_csv(f"{OUT}/X_reference_scenarios.csv", index_col=0)   # filler-corrected see/do reference from run 1b
print(f"{len(EV)} items x {len(X_LAYERS) * len(MODES) * len(ALPHAS_X) * len(MAPS) * len(ALL_PROBES)} settings")

In [ ]:
stories, spans = EV.base.tolist(), EV.span_base.tolist()
base_b = {p: belief2(stories, EV[q].tolist(), spans) for p, q in ALL_PROBES.items()}
t0 = time.time()
for L in X_LAYERS:
    path = f"{OUT}/Y_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for mode in MODES:
        for alpha in ALPHAS_X:
            for kind in MAPS:
                vecs = torch.stack([alpha * MAPS[kind][s][LI[L]] for s in EV.scenario_id])
                r = EV[["scenario_id", "item_id"]].copy()
                r["layer"], r["mode"], r["alpha"], r["kind"] = L, mode, alpha, kind
                for p, q in ALL_PROBES.items():
                    r[f"d{p}"] = belief2(stories, EV[q].tolist(), spans, edit=dict(layer=L, vecs=vecs, mode=mode)) - base_b[p]
                rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Results: which settings pass, and what D they give

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/Y_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "mode", "alpha", "kind"])[["dcause", "deffect", "dmed", "dfill", "dcauseC", "deffectC", "dmedC"]].mean().reset_index()
g.to_csv(f"{OUT}/Y_scenario_means.csv", index=False)

rows = []
for (L, mode, a), grp in g.groupby(["layer", "mode", "alpha"]):
    re = grp[grp.kind == "real"].set_index("scenario_id"); un = grp[grp.kind == "unrelated"].set_index("scenario_id")
    diff = (re.dmedC - un.dmedC)
    p = wilcoxon(diff).pvalue if (diff != 0).sum() > 5 else np.nan
    rows.append(dict(layer=L, mode=mode, alpha=a, dmedC_real=re.dmedC.mean(), dmedC_unrel=un.dmedC.mean(),
                     real_minus_unrel=diff.mean(), share_real_wins=(diff > 0).mean(), wilcoxon_p=p,
                     dfill_real=re.dfill.mean(), dcauseC_real=re.dcauseC.mean(), deffectC_real=re.deffectC.mean(),
                     passes=(re.dmedC.mean() >= 1.0) and (diff.mean() > 0.5) and ((diff > 0).mean() >= 0.7) and (abs(re.dfill.mean()) < 0.5)))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/Y_settings.csv", index=False)
display(res.round(2))
PASS = res[res.passes]
print(f"\n{len(PASS)} of {len(res)} settings pass:", PASS[["layer", "mode", "alpha"]].values.tolist())

# D for passing settings (filler-corrected; same matched-effect rule as before)
real = g[g.kind == "real"].merge(xsc[["deffectC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
real["works"] = real.deffectC >= 0.5 * real.deffectC_see
real["r_edit"] = real.dcauseC / real.deffectC
real["D"] = np.where(real.works, (real.r_see - real.r_edit) / (real.r_see - real.r_do), np.nan)
real.to_csv(f"{OUT}/Y_scenario_scores.csv", index=False)
out = []
for (L, mode, a), grp in real.groupby(["layer", "mode", "alpha"]):
    lo, hi = boot_median(grp.D)
    out.append(dict(layer=L, mode=mode, alpha=a, works_rate=grp.works.mean(), n_scored=int(grp.D.notna().sum()),
                    D_median=grp.D.median(), ci_lo=lo, ci_hi=hi))
dres = pd.DataFrame(out).merge(res[["layer", "mode", "alpha", "passes"]], on=["layer", "mode", "alpha"])
dres.to_csv(f"{OUT}/Y_D_summary.csv", index=False)
print("D (filler-corrected) for every setting; trust only rows with passes = True and n_scored >= 8")
display(dres.round(2))